### Filter Relevant Data

In [ ]:
import json
import pandas as pd

COLUMNS_SELECTION = ["id", "date_posted", "date_created", "title", "description_text", "seniority", "url", "countries_derived", "locations_derived", "organization", "organization_logo", "linkedin_org_url"]

with open("data/linkedin_api.json") as f:
    data = json.load(f)       
    df = pd.DataFrame(data)
    selected_columns_df = df[COLUMNS_SELECTION] 

# Give more weight to job postings similar to each other and avoid cluttering database with its duplicates.
selected_columns_df["weight"] = selected_columns_df.groupby(["organization", "title", "description_text"])["id"].transform("count")    
filtered_df = selected_columns_df.drop_duplicates(subset=["organization", "title", "description_text"])

In [13]:
filtered_df

,id,date_posted,date_created,title,description_text,seniority,url,countries_derived,locations_derived,organization,organization_logo,linkedin_org_url,weight
0,2124337218,2026-04-25T11:14:58.445,2026-04-25T11:14:58.514779,Data Engineer,Data Engineer (remoto)\n\nConjunto de habilida...,Assistente,https://br.linkedin.com/jobs/view/data-enginee...,[Brazil],[Brazil],Pantheon Inc,https://media.licdn.com/dms/image/v2/C4D0BAQGF...,http://www.pantheon-inc.com,1
1,2124334674,2026-04-25T11:13:52.415,2026-04-25T11:13:52.490391,Data Engineer (Remote),Role: Data Engineer (Remote)\nLocation: Remote...,Não aplicável,https://br.linkedin.com/jobs/view/data-enginee...,[Brazil],[Brazil],Jobs Ai,https://media.licdn.com/dms/image/v2/D560BAQEe...,None,1
2,2124334619,2026-04-25T11:13:51.078,2026-04-25T11:13:51.161643,Data Engineer | SR AWS / Spark (Remote),Responsabilidades e atribuições\n\nDesenvolver...,Não aplicável,https://br.linkedin.com/jobs/view/data-enginee...,[Brazil],[Brazil],Compass UOL,https://media.licdn.com/dms/image/v2/D4D0BAQGS...,https://aircompany.ai,1
3,2124334408,2026-04-25T11:13:46.134,2026-04-25T11:13:46.211244,Graduate Data Engineer,"About Joveo:\nEvery company says they're ""AI-f...",Não aplicável,https://br.linkedin.com/jobs/view/graduate-dat...,[Brazil],[Brazil],Joveo AI,https://media.licdn.com/dms/image/v2/D560BAQGZ...,https://www.joveo.com,1
4,2123771365,2026-04-25T01:13:44.63,2026-04-25T01:13:44.702719,Data Platform Engineer Senior,Se você tem paixão por inovação e busca trabal...,Não aplicável,https://br.linkedin.com/jobs/view/data-platfor...,[Brazil],[Brazil],Asaas,https://media.licdn.com/dms/image/v2/D4D0BAQHn...,https://www.asaas.com,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...
495,2028612640,2026-02-24T19:00:52,2026-02-24T21:13:50.344889,Data Engineer (Medior Analyst),Requisition ID: 70503\n\nAbout Whirlpool Corpo...,Pleno-sênior,https://br.linkedin.com/jobs/view/data-enginee...,[Brazil],"[São Paulo, São Paulo, Brazil]",Whirlpool Corporation,https://media.licdn.com/dms/image/v2/C560BAQEc...,http://www.whirlpoolcorp.com,1
496,2028612639,2026-02-24T17:53:32,2026-02-24T21:13:50.326716,Senior Data Scientist / Machine Learning Engineer,Data Scientist / ML Engineer\n\nWho We Are\n\n...,Pleno-sênior,https://br.linkedin.com/jobs/view/senior-data-...,[Brazil],"[São Paulo, São Paulo, Brazil]",TELUS Digital,https://media.licdn.com/dms/image/v2/D560BAQG9...,https://www.telusdigital.com,1
497,2117628882,2026-02-24T02:33:23,2026-04-21T21:14:27.324696,Data Engineer,Your Profile\n\n7 years of experience in data ...,Pleno-sênior,https://br.linkedin.com/jobs/view/data-enginee...,[Brazil],"[Fundão, Espírito Santo, Brazil]",Capgemini,https://media.licdn.com/dms/image/v2/D4E0BAQHI...,https://www.capgemini.com,1
498,2025382368,2026-02-23T15:21:08,2026-02-23T16:04:53.235409,Data Engineer Pleno,Descrição\n\nO time de engenharia de dados da ...,Pleno-sênior,https://br.linkedin.com/jobs/view/data-enginee...,[Brazil],"[Porto Alegre, Rio Grande do Sul, Brazil]",EvcomX,https://media.licdn.com/dms/image/v2/D4D0BAQGp...,https://evcomx.com.br/,1


### Transform data

In [28]:
renamed_columns = {
    "description_text": "description",
    "countries_derived": "country",
    "locations_derived": "location"
}
country = lambda x: x["country"][0] if x["country"] else None
location = lambda x: x["location"][0] if x["location"] else None

transformed_df = filtered_df.rename(columns=renamed_columns)

transformed_df["country"] = transformed_df.apply(country, axis=1)
transformed_df["location"] = transformed_df.apply(location, axis=1)

transformed_df["date_posted"] = transformed_df["date_posted"].apply(lambda x: x.split("T")[0] if x else None)
transformed_df["date_created"] = transformed_df["date_created"].apply(lambda x: x.split("T")[0] if x else None)

renamed_seniority = {
    "Pleno-sênior": "Mid",
    "Não aplicável": None,
    "Assistente": "Junior",
    "Júnior": "Junior",
    "Cadre": None,
    "Non pertinent": None,
    "Mid-Senior level": "Mid",
    "Directeur": "Director",
    "Confirmé": None,
    "Associate": "Associate",
    "Estagiário": "Intern",
    "Estágio": "Intern"
}
transformed_df.replace({"seniority": renamed_seniority}, inplace=True)

transformed_df["c_source"] = "LinkedIn API"
transformed_df["f_ai_min_seniority"] = transformed_df["seniority"]
transformed_df["ai_experience_time_months"] = None
transformed_df["ai_industries"] = None  # List of industries extracted from job description using AI

del transformed_df["seniority"]

In [35]:
# transformed_df["ai_hard_skills"] = None  # List of hard skills extracted from job description using AI
# transformed_df["ai_hard_skills_vectors"] = None  # Vector representations of hard skills for similarity analysis
# transformed_df
transformed_df.iloc[:3, :].to_dict(orient="list")

{'id': ['2124337218', '2124334674', '2124334619'],
 'date_posted': ['2026-04-25', '2026-04-25', '2026-04-25'],
 'date_created': ['2026-04-25', '2026-04-25', '2026-04-25'],
 'title': ['Data Engineer',
  'Data Engineer (Remote)',
  'Data Engineer | SR AWS / Spark (Remote)'],
 'description': ['Data Engineer (remoto)\n\nConjunto de habilidades principais (habilidades técnicas obrigatórias):\n\nSólido conhecimento em SQL\nDesenvolvimento de soluções de integração de dados e ETL (preferencialmente em plataformas Microsoft – SSIS ou Azure)\nFerramentas de Business Intelligence (preferencialmente em plataformas Microsoft – Power BI, SSRS, SSAS ou Tableau)\n\nHabilidades desejáveis:\n\nLinguagem de programação (Python é preferencial)\nFundamentos de nuvem (Azure ou AWS)\nArquiteturas de pipeline de dados e análise\nInglês fluente (escrito e falado)\n\nHabilidades desejáveis:\n\nSnowflake\nAzure Data Factory\nBODS\nFundamentos de DevOps',
  'Role: Data Engineer (Remote)\nLocation: Remote\nPayout

### Save data to local database

In [ ]:
import psycopg2
from os import getenv

database_user = getenv("DB_USER")
database_password = getenv("DB_PASSWORD")
HOST = "localhost"
DATABASE = "market_fit"
TABLE_NAME = "jobs"

conn = psycopg2.connect(
    host=HOST,
    database=DATABASE,
    user=database_user,
    password=database_password
)

cur = conn.cursor()


NameError: name 'getenv' is not defined